In this notebook we assemble gene cohorts from condition cohorts

In [ ]:
import pandas as pd
from pathlib import Path

bucket = Path.home() / 'workspace/scalable-vus-validation'
condition_cohorts_dir = bucket / 'condition_cohorts'
gene_cohorts_dir = bucket / 'gene_cohorts'

In [ ]:
gene_cohorts_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Mapping of genes to phenotypes:
# gene: (case phenotypes, control phenotypes, inheritence)
gene_phenotypes = {
    'BAP1': (
        {
            'melanoma',
            'mesothelioma',
            'renal cancer',
            'basal cell carcinoma',
            'intracranial meningioma',
            'cholangiocarcinoma of the biliary tract'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'BARD1': (
        {'breast cancer', 'ovarian cancer'},
        {'cancer'},
        'autosomal dominant'
    ),
    'BRCA1': (
        {
            'breast cancer',
            'ovarian cancer',
            'prostate cancer',
            'pancreatic cancer',
            'melanoma'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'BRCA2': (
        {
            'breast cancer',
            'ovarian cancer',
            'prostate cancer',
            'pancreatic cancer',
            'melanoma'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'CHEK2': (
        {
            'breast cancer',
            'prostate cancer'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'GCK': (
        {'MODY'},
        {'diabetes'},
        'autosomal dominant'
    ),
    'KCNE1': (
        {'long QT syndrome', 'sudden cardiac death'},
        {'abnormal QT interval', 'cardiac arrhythmia', 'sudden cardiac death'},
        'autosomal dominant'
    ),
    'KCNH2': (
        {'long QT syndrome', 'sudden cardiac death'},
        {'abnormal QT interval', 'cardiac arrhythmia', 'sudden cardiac death'},
        'autosomal dominant'
    ),
    'KCNQ4': (
        {'nonsyndromic genetic hearing loss'},
        {'hearing loss'},
        'autosomal dominant'
    ),
    'MSH2': (
        {
            'colorectal cancer',
            'ovarian cancer',
            'stomach cancer',
            'small bowel cancer',
            'urinary tract cancer',
            'prostate cancer',
            'brain cancer',
            'endometrial cancer'
        },
        {'cancer'},
        'autosomal dominant' # AR for mismatch repair- maybe we should also consider AR analysis
    ),
    'PALB2': (
        {
            'breast cancer',
            'ovarian cancer',
            'pancreatic cancer',
            'prostate cancer'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'PTEN': (
        {
            'breast cancer',
            'endometrial cancer',
            'thyroid cancer',
            'colorectal cancer',
            'renal cancer',
            'melanoma',
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'RAD51C': (
        {
            'breast cancer',
            'ovarian cancer',
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'RAD51D': (
        {
            'breast cancer',
            'ovarian cancer',
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'SCN5A':(
        {
            'long QT syndrome',
            'dilated cardiomyopathy',
            'sudden cardiac death'
        },
        {
            'abnormal QT interval',
            'cardiomyopathy',
            'cardiac arrhythmia',
            'sudden cardiac death'
        },
        'autosomal dominant'
    ),
    'TP53': (
        {'cancer'},
        {'cancer'},
        'autosomal dominant'
    ),
    'TSC2': (
        {'tuberous sclerosis'},
        {'tuberous sclerosis', 'cancer'},
        'autosomal dominant'
    )
}

In [ ]:
for gene, (case_phenos, control_phenos, _) in gene_phenotypes.items():

    print(f'Assembing cohort for {gene}...')

    out_path = Path(gene_cohorts_dir, f'{gene} cohort.parquet')
    if out_path.exists():
        print(f'{out_path} exists, skipping.')
        continue

    try:
        case_dfs = [
            pd.read_table(
                f'{condition_cohorts_dir}/{case_cohort} case pool.tsv.gz',
                usecols=['person_id'],
                index_col='person_id'
            ).assign(case=1)
            for case_cohort in case_phenos
        ]
    
        controls = None
        for control_cohort in control_phenos:
            df = pd.read_table(f'{condition_cohorts_dir}/{control_cohort} control pool.tsv.gz',
                usecols=['person_id'],
                index_col='person_id'
            )
            if controls is None:
                controls = df
            else:
                controls = controls.join(df, how='inner')
    except FileNotFoundError as e:
        print(e)
        continue
    
    gene_cohort = pd.concat(
        case_dfs + [controls.assign(case=0)]
    )

    gene_cohort.to_parquet(out_path)
    print(f'Saved to {out_path}')